In [1]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ============================================================
# 1. Base paths
# ============================================================
BASE_DIR = Path("../data")

MIRAGE_DIR = BASE_DIR / "MiRAGE"
PRIMEKG_DIR = BASE_DIR / "kg"

# MiRAGE / Kaggle drug repositioning files
MIRAGE_MAPPING_PATH = MIRAGE_DIR / "mapping.csv"
MIRAGE_DRUGINFO_PATH = MIRAGE_DIR / "drugsinfo.csv"
MIRAGE_DISEASEINFO_PATH = MIRAGE_DIR / "diseasesinfo.csv"

# PrimeKG files
PRIMEKG_NODES_PATH = PRIMEKG_DIR / "node.csv"
PRIMEKG_KG_PATH = PRIMEKG_DIR / "kg.csv"
PRIMEKG_EDGES_PATH = PRIMEKG_DIR / "edges.csv"
PRIMEKG_KG_DIRECTED_PATH = PRIMEKG_DIR / "kg_directed.csv"

# MONDO ontology file used for MeSH -> MONDO disease mapping
MONDO_JSON_PATH = MIRAGE_DIR / "mondo.json"

# Output directory
OUTPUT_DIR = BASE_DIR / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. LLM training source paths
# ============================================================
# LLM 训练材料来自 RepoDB 和 DrugRepoBank 的原始 drug-disease pairs。
# 后续将用 DrugBank ID + disease name 文本标准化，从 MiRAGE 外部测试集中
# 剔除与这些训练来源重复的关系。
REPODB_RAW_PATH = BASE_DIR / "repodb/repodb.csv"
DRUGREPOBANK_RAW_PATH = BASE_DIR / "drugrepobank/drugrepobank.csv"

# 是否启用 raw source overlap 过滤。
USE_RAW_TRAIN_SOURCE_OVERLAP_FILTER = True

# ============================================================
# 3. PrimeKG relation settings
# ============================================================
# MiRAGE 是 positive-only 外部数据。
# 为避免测试泄漏，凡是 MiRAGE pair 已经出现在 PrimeKG 的 drug-disease
# therapeutic/safety relations 中，就从 external test 中剔除。
PRIMEKG_DD_RELATIONS = {
    "indication",
    "off-label use",
    "contraindication",
    "rev_indication",
    "rev_off-label use",
    "rev_contraindication",
}

# MiRAGE positive-only pair 转成 TxGNN external positive edge 时使用的默认关系。
DEFAULT_EXTERNAL_RELATION = "indication"

# Reproducibility
SEED = 42

print("Configuration loaded.")
print(f"BASE_DIR: {BASE_DIR.resolve()}")
print(f"MIRAGE_DIR: {MIRAGE_DIR.resolve()}")
print(f"PRIMEKG_DIR: {PRIMEKG_DIR.resolve()}")
print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()}")
print(f"RepoDB raw path: {REPODB_RAW_PATH.resolve()}")
print(f"DrugRepoBank raw path: {DRUGREPOBANK_RAW_PATH.resolve()}")

Configuration loaded.
BASE_DIR: D:\PycharmProjects\TxGNN\data
MIRAGE_DIR: D:\PycharmProjects\TxGNN\data\MiRAGE
PRIMEKG_DIR: D:\PycharmProjects\TxGNN\data\kg
OUTPUT_DIR: D:\PycharmProjects\TxGNN\data\processed
RepoDB raw path: D:\PycharmProjects\TxGNN\data\repodb\repodb.csv
DrugRepoBank raw path: D:\PycharmProjects\TxGNN\data\drugrepobank\drugrepobank.csv


In [2]:
# ============================================================
# 2. Utility functions: ID normalization, text normalization,
#    robust CSV reading, and encoding test
# ============================================================

def normalize_id(x):
    """
    Normalize general string IDs.
    Handles NaN, extra spaces, and pandas-read numeric artifacts like '123.0'.
    """
    if pd.isna(x):
        return pd.NA

    s = str(x).strip()
    if s == "":
        return pd.NA

    # Avoid turning IDs like DB00123 into numbers.
    # Only normalize pure numeric strings such as '123.0'.
    try:
        f = float(s)
        if f.is_integer() and re.fullmatch(r"\d+(\.0+)?", s):
            return str(int(f))
    except Exception:
        pass

    return s


def canonical_drugbank_id(x):
    """
    Normalize DrugBank IDs.
    Example: ' db00123 ' -> 'DB00123'
    """
    s = normalize_id(x)
    if pd.isna(s):
        return pd.NA
    return str(s).strip().upper()


def canonical_mesh_id(x, with_prefix=True):
    """
    Normalize MeSH disease IDs.

    Examples:
        'MESH:D000013' -> 'MESH:D000013' if with_prefix=True
        'D13'          -> 'MESH:D000013'
        '000013'       -> 'MESH:D000013'
        'D000013'      -> 'D000013' if with_prefix=False
    """
    if pd.isna(x):
        return pd.NA

    s = str(x).strip()
    if s == "":
        return pd.NA

    s = s.replace("MESH:", "")
    s = s.split("/")[-1]

    if s.startswith("D"):
        num = s[1:]
    else:
        num = s

    num = re.sub(r"\D", "", num)
    if num == "":
        return pd.NA

    mesh = "D" + num.zfill(6)
    return "MESH:" + mesh if with_prefix else mesh


def normalize_primekg_node_id(x):
    """
    Normalize PrimeKG node_id.

    PrimeKG disease MONDO node IDs in your earlier notebook appear as numeric strings
    with prefixes and leading zeros removed. This function standardizes:
        'MONDO:0001234' -> '1234'
        'MONDO_0001234' -> '1234'
        '1234.0'        -> '1234'
    DrugBank IDs such as DB00001 are preserved.
    """
    s = normalize_id(x)
    if pd.isna(s):
        return pd.NA

    s = str(s).strip()

    if s.startswith("MONDO:"):
        s = s.replace("MONDO:", "")
    if s.startswith("MONDO_"):
        s = s.replace("MONDO_", "")

    if re.fullmatch(r"\d+(\.0+)?", s):
        return str(int(float(s)))

    return s


def normalize_text(x):
    """
    Normalize text for conservative name-level overlap matching.
    """
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


def safe_read_csv(path, encodings=("utf-8", "utf-8-sig", "cp1252", "latin1", "gb18030"), **kwargs):
    """
    Try reading a CSV using multiple common encodings.
    Returns:
        df, encoding_used
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    last_error = None
    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc, **kwargs)
            return df, enc
        except Exception as e:
            last_error = e

    raise RuntimeError(f"Failed to read {path} with encodings={encodings}. Last error: {last_error}")


def try_read_csv_encodings(path, encodings=("utf-8", "utf-8-sig", "cp1252", "latin1", "gb18030"), nrows=5):
    """
    Print which encodings can read the CSV. Useful for files like DrugRepoBank.
    """
    path = Path(path)
    print(f"Testing encodings for: {path}")
    print("=" * 100)

    results = []

    for enc in encodings:
        try:
            df_tmp = pd.read_csv(path, encoding=enc, nrows=nrows)
            print(f"[OK] encoding={enc}, shape={df_tmp.shape}")
            display(df_tmp.head(2))
            results.append((enc, "OK", None))
        except Exception as e:
            print(f"[FAIL] encoding={enc}: {type(e).__name__}: {e}")
            results.append((enc, "FAIL", str(e)))
        print("-" * 100)

    return results


def report_count(name, df):
    print(f"{name}: {len(df):,}")

In [3]:
# ============================================================
# 3. Load raw data
# ============================================================

# MiRAGE files
mapping_df, mapping_enc = safe_read_csv(MIRAGE_MAPPING_PATH)
drug_info, druginfo_enc = safe_read_csv(MIRAGE_DRUGINFO_PATH)
disease_info, diseaseinfo_enc = safe_read_csv(MIRAGE_DISEASEINFO_PATH)

# PrimeKG nodes
node_df, node_enc = safe_read_csv(PRIMEKG_NODES_PATH)

# LLM training source raw files
repodb_df, repodb_enc = safe_read_csv(REPODB_RAW_PATH)

# DrugRepoBank: known to contain non-UTF-8-compatible bytes; use cp1252.
drugrepobank_df = pd.read_csv(DRUGREPOBANK_RAW_PATH, encoding="cp1252")
drugrepobank_enc = "cp1252"

print("Loaded files and encodings:")
print(f"mapping.csv:       {mapping_enc}, shape={mapping_df.shape}")
print(f"druginfo.csv:      {druginfo_enc}, shape={drug_info.shape}")
print(f"diseaseinfo.csv:   {diseaseinfo_enc}, shape={disease_info.shape}")
print(f"PrimeKG nodes.csv: {node_enc}, shape={node_df.shape}")
print(f"repodb.csv:        {repodb_enc}, shape={repodb_df.shape}")
print(f"drugrepobank.csv:  {drugrepobank_enc}, shape={drugrepobank_df.shape}")

print("\nColumns:")
print("mapping_df:", mapping_df.columns.tolist())
print("drug_info:", drug_info.columns.tolist())
print("disease_info:", disease_info.columns.tolist())
print("node_df:", node_df.columns.tolist())
print("repodb_df:", repodb_df.columns.tolist())
print("drugrepobank_df:", drugrepobank_df.columns.tolist())

# ============================================================
# Required column checks
# ============================================================

required_mapping_cols = {"DrugID", "DiseaseID"}
required_druginfo_cols = {"DrugID"}
required_diseaseinfo_cols = {"DiseaseID"}
required_nodes_cols = {"node_index", "node_id", "node_type", "node_name"}

missing_mapping = required_mapping_cols - set(mapping_df.columns)
missing_drug_info = required_druginfo_cols - set(drug_info.columns)
missing_disease_info = required_diseaseinfo_cols - set(disease_info.columns)
missing_nodes = required_nodes_cols - set(node_df.columns)

assert not missing_mapping, f"mapping.csv missing columns: {missing_mapping}"
assert not missing_drug_info, f"druginfo.csv missing columns: {missing_drug_info}"
assert not missing_disease_info, f"diseaseinfo.csv missing columns: {missing_disease_info}"
assert not missing_nodes, f"PrimeKG nodes.csv missing columns: {missing_nodes}"

# RepoDB expected columns
required_repodb_cols = {"drugbank_id", "ind_name"}
missing_repodb = required_repodb_cols - set(repodb_df.columns)
assert not missing_repodb, f"repodb.csv missing columns: {missing_repodb}"

# DrugRepoBank expected columns
required_drb_cols = {"DrugID"}
missing_drb = required_drb_cols - set(drugrepobank_df.columns)
assert not missing_drb, f"drugrepobank.csv missing columns: {missing_drb}"

# ============================================================
# ID normalization
# ============================================================

# MiRAGE pair IDs
mapping_df["DrugID"] = mapping_df["DrugID"].apply(canonical_drugbank_id)
mapping_df["DiseaseID"] = mapping_df["DiseaseID"].apply(
    lambda x: canonical_mesh_id(x, with_prefix=True)
)

# MiRAGE text info IDs
drug_info["DrugID"] = drug_info["DrugID"].apply(canonical_drugbank_id)
disease_info["DiseaseID"] = disease_info["DiseaseID"].apply(
    lambda x: canonical_mesh_id(x, with_prefix=True)
)

# PrimeKG node IDs
node_df["node_id"] = node_df["node_id"].apply(normalize_primekg_node_id)
node_df["node_type"] = node_df["node_type"].astype(str)

# RepoDB IDs / names
repodb_df["drugbank_id"] = repodb_df["drugbank_id"].apply(canonical_drugbank_id)
repodb_df["ind_name_norm"] = repodb_df["ind_name"].apply(normalize_text)

# DrugRepoBank IDs
drugrepobank_df["DrugID"] = drugrepobank_df["DrugID"].apply(canonical_drugbank_id)

# Disease name fields in DrugRepoBank:
# - Disease: original disease/context
# - NewDisease: repositioning/new disease, likely more relevant
if "Disease" in drugrepobank_df.columns:
    drugrepobank_df["Disease_norm"] = drugrepobank_df["Disease"].apply(normalize_text)
else:
    drugrepobank_df["Disease_norm"] = ""

if "NewDisease" in drugrepobank_df.columns:
    drugrepobank_df["NewDisease_norm"] = drugrepobank_df["NewDisease"].apply(normalize_text)
else:
    drugrepobank_df["NewDisease_norm"] = ""

# ============================================================
# Basic summary
# ============================================================

print("\nBasic counts:")
report_count("MiRAGE mapping rows", mapping_df)
print("MiRAGE unique DrugID:", mapping_df["DrugID"].nunique())
print("MiRAGE unique DiseaseID:", mapping_df["DiseaseID"].nunique())

print("\nMiRAGE mapping head:")
display(mapping_df.head())

print("\nDrugRepoBank head:")
display(drugrepobank_df.head())

print("\nRepoDB head:")
display(repodb_df.head())

Loaded files and encodings:
mapping.csv:       utf-8, shape=(42200, 2)
druginfo.csv:      utf-8, shape=(1410, 9)
diseaseinfo.csv:   utf-8, shape=(1573, 5)
PrimeKG nodes.csv: utf-8, shape=(129375, 5)
repodb.csv:        utf-8, shape=(13558, 8)
drugrepobank.csv:  cp1252, shape=(169, 16)

Columns:
mapping_df: ['DrugID', 'DiseaseID']
drug_info: ['DrugID', 'DrugName', 'DrugDescription', 'DrugTarget', 'DrugPharmacodynamics', 'DrugSmile', 'DrugMechanism', 'DrugConditions', 'DrugCategories']
disease_info: ['DiseaseID', 'DiseaseName', 'DiseaseDescription', 'SlimMapping', 'PathwayNames']
node_df: ['node_index', 'node_id', 'node_type', 'node_name', 'node_source']
repodb_df: ['drug_name', 'drugbank_id', 'ind_name', 'ind_id', 'NCT', 'status', 'phase', 'DetailedStatus']
drugrepobank_df: ['DrugName', 'DrugID', 'PubChemID', 'Target', 'Disease', 'Side_effect', 'NewDirectTarget', 'NewIndirectTarget', 'NewDisease', 'Evidence', 'Insilico', 'Invitro', 'Invivo', 'Clinicaltrial', 'SupportedSentences', 'PMID']

,DrugID,DiseaseID
0,DB09140,MESH:D000013
1,DB00730,MESH:D000013
2,DB00898,MESH:D000013
3,DB01168,MESH:D000013
4,DB00550,MESH:D000013



DrugRepoBank head:


,DrugName,DrugID,PubChemID,Target,Disease,Side_effect,NewDirectTarget,NewIndirectTarget,NewDisease,Evidence,Insilico,Invitro,Invivo,Clinicaltrial,SupportedSentences,PMID,Disease_norm,NewDisease_norm
0,Spiramycin,DB06145,6440717.0,Gram-positive bacteria,Bacterial infection,NaN,NaN,LPS; iNOS; MAPKs; ERK; JNK; NF-?oB,Anti-inflammatory,Human skin primary irritation tests,NaN,Human skin primary irritation tests,NaN,NaN,Anti-Inflammatory Effects of Spiramycin in LPS...,35630676,bacterial infection,anti-inflammatory
1,Disulfiram,DB00822,3117.0,AL3A2,Alcohol dependence,NaN,NaN,metastasis related proteins (PCNA; MMP-2); Wnt...,Gastric cancer,Transwell assays; ELISA,NaN,Transwell assays; ELISA,NaN,NaN,The anti-alcohol dependency drug disulfiram in...,32529870,alcohol dependence,gastric cancer
2,Fluvoxamine,DB00176,3404.0,Serotonin system,Depression,NaN,NaN,A?242,Alzheimer's disease (AD),Aggregation kinetic experiments; transmission ...,NaN,Aggregation kinetic experiments; Transmission ...,NaN,NaN,These studies demonstrate that SSRIs have the ...,30157623,depression,alzheimer's disease (ad)
3,Paroxetine,DB00715,43815.0,SC6A4,Depression,NaN,NaN,A?242,Alzheimer's disease (AD),Aggregation kinetic experiments; transmission ...,NaN,Aggregation kinetic experiments; Transmission ...,NaN,NaN,These studies demonstrate that SSRIs have the ...,30157623,depression,alzheimer's disease (ad)
4,Fluoxetine,DB00472,3386.0,SC6A4,Depression,NaN,NaN,A?242,Alzheimer's disease (AD),Aggregation kinetic experiments; transmission ...,NaN,Aggregation kinetic experiments; Transmission ...,NaN,NaN,These studies demonstrate that SSRIs have the ...,30157623,depression,alzheimer's disease (ad)



RepoDB head:


,drug_name,drugbank_id,ind_name,ind_id,NCT,status,phase,DetailedStatus,ind_name_norm
0,ajmaline,DB01426,Ventricular arrhythmia,C0085612,NaN,Approved,NaN,NaN,ventricular arrhythmia
1,ajmaline,DB01426,Supraventricular arrhythmia,C0428974,NaN,Approved,NaN,NaN,supraventricular arrhythmia
2,ajmaline,DB01426,Cardiac Arrhythmia,C0003811,NaN,Approved,NaN,NaN,cardiac arrhythmia
3,emtricitabine,DB00879,HIV Infections,C0019693,NaN,Approved,NaN,NaN,hiv infections
4,enalapril,DB00584,Asymptomatic left ventricular systolic dysfunc...,C3698411,NaN,Approved,NaN,NaN,asymptomatic left ventricular systolic dysfunc...


In [4]:
# ============================================================
# 4. Merge MiRAGE mapping with drug/disease text information
# ============================================================

mirage = mapping_df.copy()

# MiRAGE 是 positive-only 数据集；如果原始 mapping.csv 没有 label，则统一补 label=1。
if "label" not in mirage.columns:
    mirage["label"] = 1

# 合并 drug text fields
mirage = mirage.merge(
    drug_info,
    on="DrugID",
    how="left",
    validate="many_to_one",
)

# 合并 disease text fields
mirage = mirage.merge(
    disease_info,
    on="DiseaseID",
    how="left",
    validate="many_to_one",
)

# 文本标准化列，后续用于和 RepoDB / DrugRepoBank 做 name-level overlap 过滤
mirage["DrugName_norm"] = mirage["DrugName"].apply(normalize_text)
mirage["DiseaseName_norm"] = mirage["DiseaseName"].apply(normalize_text)

# pair key: 用于 MiRAGE 内部去重和训练源 overlap 检查
mirage["drug_disease_name_key"] = list(
    zip(mirage["DrugID"], mirage["DiseaseName_norm"])
)

# ============================================================
# Basic quality checks
# ============================================================

print("MiRAGE with text info shape:", mirage.shape)

missing_drug_name = mirage["DrugName"].isna().sum()
missing_disease_name = mirage["DiseaseName"].isna().sum()

print(f"Missing DrugName rows: {missing_drug_name:,} ({missing_drug_name / len(mirage):.4%})")
print(f"Missing DiseaseName rows: {missing_disease_name:,} ({missing_disease_name / len(mirage):.4%})")

print("\nLabel distribution:")
print(mirage["label"].value_counts(dropna=False))

print("\nDuplicate DrugID-DiseaseID pairs:")
dup_count = mirage.duplicated(subset=["DrugID", "DiseaseID"]).sum()
print(f"Duplicated rows by DrugID-DiseaseID: {dup_count:,}")

print("\nMiRAGE merged head:")
display(mirage.head())

# Optional: if duplicate pairs exist, inspect them.
if dup_count > 0:
    print("\nExample duplicated pairs:")
    display(
        mirage[mirage.duplicated(subset=["DrugID", "DiseaseID"], keep=False)]
        .sort_values(["DrugID", "DiseaseID"])
        .head(10)
    )

MiRAGE with text info shape: (42200, 18)
Missing DrugName rows: 0 (0.0000%)
Missing DiseaseName rows: 0 (0.0000%)

Label distribution:
label
1    42200
Name: count, dtype: int64

Duplicate DrugID-DiseaseID pairs:
Duplicated rows by DrugID-DiseaseID: 0

MiRAGE merged head:


,DrugID,DiseaseID,label,DrugName,DrugDescription,DrugTarget,DrugPharmacodynamics,DrugSmile,DrugMechanism,DrugConditions,DrugCategories,DiseaseName,DiseaseDescription,SlimMapping,PathwayNames,DrugName_norm,DiseaseName_norm,drug_disease_name_key
0,DB09140,MESH:D000013,1,Oxygen,Oxygenis an essential element for human surviv...,"['P00395', 'Q9Y5S8']",Oxygen therapy improves effective cellular oxy...,O=O,Oxygen therapy increases the arterial pressure...,[],"['Chalcogens', 'Elements', 'Gases', 'Medical G...",Congenital Abnormalities,Malformations of organs or body parts during d...,['Congenital abnormality'],Vesicle-mediated transport,oxygen,congenital abnormalities,"(DB09140, congenital abnormalities)"
1,DB00730,MESH:D000013,1,Thiabendazole,Thiabendazoleis a benzimidazole used in the tr...,['P00363'],Thiabendazole is a fungicide and parasiticide....,N1C2=CC=CC=C2N=C1C1=CSC=N1,The precise mode of action of thiabendazole on...,[],"['Anthelmintics', 'Anti-Infective Agents', 'An...",Congenital Abnormalities,Malformations of organs or body parts during d...,['Congenital abnormality'],Vesicle-mediated transport,thiabendazole,congenital abnormalities,"(DB00730, congenital abnormalities)"
2,DB00898,MESH:D000013,1,Ethanol,"A clear, colorless liquid rapidly absorbed fro...","['P14867', 'Q8TCU5', 'P23415', 'P23416', 'Q026...",Alcohol produces injury to cells by dehydratio...,CCO,Ethanol affects the brain’s neurons in several...,"['Hand Hygiene', 'Skin disinfection']","['Agents Causing Muscle Toxicity', 'Alcohols',...",Congenital Abnormalities,Malformations of organs or body parts during d...,['Congenital abnormality'],Vesicle-mediated transport,ethanol,congenital abnormalities,"(DB00898, congenital abnormalities)"
3,DB01168,MESH:D000013,1,Procarbazine,Procarbazineis an antineoplastic agent indicat...,"['P21397', 'P27338']",Procarbazine is an antineoplastic in the class...,CNNCC1=CC=C(C=C1)C(=O)NC(C)C,The precise mode of cytotoxic action of procar...,[],"['Acids, Carbocyclic', 'Agents Causing Muscle ...",Congenital Abnormalities,Malformations of organs or body parts during d...,['Congenital abnormality'],Vesicle-mediated transport,procarbazine,congenital abnormalities,"(DB01168, congenital abnormalities)"
4,DB00550,MESH:D000013,1,Propylthiouracil,Propylthiouracilis a thiourea antithyroid agen...,['P07202'],Propylthiouracil is a thiourea antithyroid age...,CCCC1=CC(=O)NC(=S)N1,Propylthiouracil binds to thyroid peroxidase a...,[],"['Agents Causing Muscle Toxicity', 'Antimetabo...",Congenital Abnormalities,Malformations of organs or body parts during d...,['Congenital abnormality'],Vesicle-mediated transport,propylthiouracil,congenital abnormalities,"(DB00550, congenital abnormalities)"


In [5]:
# ============================================================
# 5. Map MiRAGE DrugBank IDs to PrimeKG drug nodes
# ============================================================

drug_nodes = node_df[node_df["node_type"] == "drug"].copy()

# PrimeKG drug node_id should correspond to DrugBank ID.
drug_nodes["node_id_norm"] = drug_nodes["node_id"].apply(canonical_drugbank_id)

drugid_to_primekg_node_index = drug_nodes.set_index("node_id_norm")["node_index"].to_dict()
drugid_to_primekg_node_name = drug_nodes.set_index("node_id_norm")["node_name"].to_dict()
drugid_to_primekg_node_source = drug_nodes.set_index("node_id_norm")["node_source"].to_dict()

mirage["primekg_drug_node_id"] = mirage["DrugID"].apply(canonical_drugbank_id)
mirage["primekg_drug_node_index"] = mirage["primekg_drug_node_id"].map(drugid_to_primekg_node_index)
mirage["primekg_drug_node_name"] = mirage["primekg_drug_node_id"].map(drugid_to_primekg_node_name)
mirage["primekg_drug_node_source"] = mirage["primekg_drug_node_id"].map(drugid_to_primekg_node_source)

mirage["drug_in_primekg"] = mirage["primekg_drug_node_index"].notna()

# ============================================================
# Drug mapping summary
# ============================================================

total_unique_drugs = mirage["DrugID"].nunique()
unmatched_drugids = (
    mirage.loc[~mirage["drug_in_primekg"], "DrugID"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

matched_unique_drugs = total_unique_drugs - len(unmatched_drugids)

print("Drug mapping summary:")
print(f"Total unique MiRAGE DrugID: {total_unique_drugs:,}")
print(f"Matched unique DrugID in PrimeKG: {matched_unique_drugs:,}")
print(f"Unmatched unique DrugID: {len(unmatched_drugids):,}")
print(f"Unmatched unique DrugID ratio: {len(unmatched_drugids) / total_unique_drugs:.4%}")
print(f"Unmatched DrugIDs: {unmatched_drugids}")

row_unmatched_drugs = (~mirage["drug_in_primekg"]).sum()
print(f"\nRows with unmatched drug: {row_unmatched_drugs:,} ({row_unmatched_drugs / len(mirage):.4%})")

print("\nMapped drug examples:")
display(
    mirage[
        ["DrugID", "DrugName", "primekg_drug_node_id", "primekg_drug_node_index", "primekg_drug_node_name", "drug_in_primekg"]
    ].head()
)

if row_unmatched_drugs > 0:
    print("\nUnmatched drug row examples:")
    display(
        mirage.loc[
            ~mirage["drug_in_primekg"],
            ["DrugID", "DrugName", "DiseaseID", "DiseaseName"]
        ].drop_duplicates().head(20)
    )

Drug mapping summary:
Total unique MiRAGE DrugID: 1,410
Matched unique DrugID in PrimeKG: 1,408
Unmatched unique DrugID: 2
Unmatched unique DrugID ratio: 0.1418%
Unmatched DrugIDs: ['DB04862', 'DB08845']

Rows with unmatched drug: 2 (0.0047%)

Mapped drug examples:


,DrugID,DrugName,primekg_drug_node_id,primekg_drug_node_index,primekg_drug_node_name,drug_in_primekg
0,DB09140,Oxygen,DB09140,14013.0,Oxygen,True
1,DB00730,Thiabendazole,DB00730,15307.0,Thiabendazole,True
2,DB00898,Ethanol,DB00898,14587.0,Ethanol,True
3,DB01168,Procarbazine,DB01168,14914.0,Procarbazine,True
4,DB00550,Propylthiouracil,DB00550,14640.0,Propylthiouracil,True



Unmatched drug row examples:


,DrugID,DrugName,DiseaseID,DiseaseName
2753,DB08845,Oxogluric acid,MESH:D001169,"Arthritis, Experimental"
15095,DB04862,Merimepodib,MESH:D006526,Hepatitis C


In [6]:
# ============================================================
# 6. Map MiRAGE MeSH DiseaseIDs to PrimeKG disease nodes via MONDO
# ============================================================

with open(MONDO_JSON_PATH, "r", encoding="utf-8") as f:
    mondo_json = json.load(f)

mondo_nodes = mondo_json["graphs"][0]["nodes"]


def extract_mondo_num_from_url(mondo_url):
    """
    Extract numeric MONDO ID from URLs like:
        http://purl.obolibrary.org/obo/MONDO_0001234
    Returns PrimeKG-style numeric string without leading zeros, e.g. '1234'.
    """
    if not isinstance(mondo_url, str):
        return None

    if "MONDO_" not in mondo_url:
        return None

    raw = mondo_url.split("MONDO_")[-1]
    raw = re.sub(r"\D", "", raw)

    if raw == "":
        return None

    return str(int(raw))


# Build MeSH -> set(MONDO numeric IDs)
mesh_to_mondo_nums = {}

for entry in tqdm(mondo_nodes, desc="Building MeSH -> MONDO mapping"):
    mondo_num = extract_mondo_num_from_url(entry.get("id", ""))
    if mondo_num is None:
        continue

    meta = entry.get("meta", {})
    mesh_ids = []

    # 1) xrefs: usually MESH:Dxxxxxx
    for xref in meta.get("xrefs", []):
        val = xref.get("val", "")
        if isinstance(val, str) and val.startswith("MESH:"):
            mesh = canonical_mesh_id(val, with_prefix=False)
            if not pd.isna(mesh):
                mesh_ids.append(mesh)

    # 2) basicPropertyValues exactMatch URLs, e.g. identifiers.org/mesh/Dxxxxxx
    for bp in meta.get("basicPropertyValues", []):
        pred = bp.get("pred", "")
        val = bp.get("val", "")

        if (
            isinstance(pred, str)
            and pred.endswith("exactMatch")
            and isinstance(val, str)
            and "mesh" in val.lower()
        ):
            mesh = canonical_mesh_id(val, with_prefix=False)
            if not pd.isna(mesh):
                mesh_ids.append(mesh)

    for mesh in set(mesh_ids):
        mesh_to_mondo_nums.setdefault(mesh, set()).add(mondo_num)

print(f"MeSH IDs with MONDO mapping: {len(mesh_to_mondo_nums):,}")


# PrimeKG disease nodes
disease_nodes = node_df[node_df["node_type"] == "disease"].copy()
disease_nodes["node_id_norm"] = disease_nodes["node_id"].apply(normalize_primekg_node_id)

mondo_to_primekg_node_index = disease_nodes.set_index("node_id_norm")["node_index"].to_dict()
mondo_to_primekg_node_name = disease_nodes.set_index("node_id_norm")["node_name"].to_dict()
mondo_to_primekg_node_source = disease_nodes.set_index("node_id_norm")["node_source"].to_dict()


def map_mesh_to_primekg_disease(mesh_id):
    """
    Input:
        mesh_id: MiRAGE DiseaseID, e.g. 'MESH:D000013'

    Returns:
        primekg_disease_node_id, primekg_disease_node_index, primekg_disease_node_name, primekg_disease_node_source
    """
    mesh = canonical_mesh_id(mesh_id, with_prefix=False)

    if pd.isna(mesh):
        return pd.NA, pd.NA, pd.NA, pd.NA

    if mesh not in mesh_to_mondo_nums:
        return pd.NA, pd.NA, pd.NA, pd.NA

    candidate_mondos = sorted(mesh_to_mondo_nums[mesh], key=lambda x: int(x))

    # Prefer MONDO IDs that actually exist in PrimeKG disease nodes.
    for mondo_num in candidate_mondos:
        mondo_norm = normalize_primekg_node_id(mondo_num)

        if mondo_norm in mondo_to_primekg_node_index:
            return (
                mondo_norm,
                mondo_to_primekg_node_index[mondo_norm],
                mondo_to_primekg_node_name[mondo_norm],
                mondo_to_primekg_node_source[mondo_norm],
            )

    return pd.NA, pd.NA, pd.NA, pd.NA


mapped_diseases = mirage["DiseaseID"].apply(map_mesh_to_primekg_disease)

mirage["primekg_disease_node_id"] = [x[0] for x in mapped_diseases]
mirage["primekg_disease_node_index"] = [x[1] for x in mapped_diseases]
mirage["primekg_disease_node_name"] = [x[2] for x in mapped_diseases]
mirage["primekg_disease_node_source"] = [x[3] for x in mapped_diseases]

mirage["disease_in_primekg"] = mirage["primekg_disease_node_index"].notna()

# ============================================================
# Disease mapping summary
# ============================================================

total_unique_diseases = mirage["DiseaseID"].nunique()
unmatched_diseaseids = (
    mirage.loc[~mirage["disease_in_primekg"], "DiseaseID"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

matched_unique_diseases = total_unique_diseases - len(unmatched_diseaseids)

print("\nDisease mapping summary:")
print(f"Total unique MiRAGE DiseaseID: {total_unique_diseases:,}")
print(f"Matched unique DiseaseID in PrimeKG: {matched_unique_diseases:,}")
print(f"Unmatched unique DiseaseID: {len(unmatched_diseaseids):,}")
print(f"Unmatched unique DiseaseID ratio: {len(unmatched_diseaseids) / total_unique_diseases:.4%}")

row_unmatched_diseases = (~mirage["disease_in_primekg"]).sum()
print(f"\nRows with unmatched disease: {row_unmatched_diseases:,} ({row_unmatched_diseases / len(mirage):.4%})")

print("\nFirst 50 unmatched DiseaseIDs:")
print(unmatched_diseaseids[:50])

print("\nMapped disease examples:")
display(
    mirage[
        [
            "DiseaseID",
            "DiseaseName",
            "primekg_disease_node_id",
            "primekg_disease_node_index",
            "primekg_disease_node_name",
            "disease_in_primekg",
        ]
    ].head()
)

if row_unmatched_diseases > 0:
    print("\nUnmatched disease row examples:")
    display(
        mirage.loc[
            ~mirage["disease_in_primekg"],
            ["DrugID", "DrugName", "DiseaseID", "DiseaseName", "DiseaseDescription"]
        ].drop_duplicates(subset=["DiseaseID"]).head(20)
    )

Building MeSH -> MONDO mapping:   0%|          | 0/57384 [00:00<?, ?it/s]

MeSH IDs with MONDO mapping: 8,208

Disease mapping summary:
Total unique MiRAGE DiseaseID: 1,573
Matched unique DiseaseID in PrimeKG: 861
Unmatched unique DiseaseID: 712
Unmatched unique DiseaseID ratio: 45.2638%

Rows with unmatched disease: 19,593 (46.4289%)

First 50 unmatched DiseaseIDs:
['MESH:D000015', 'MESH:D000022', 'MESH:D000067877', 'MESH:D000068079', 'MESH:D000070642', 'MESH:D000072660', 'MESH:D000075222', 'MESH:D000077192', 'MESH:D000077195', 'MESH:D000077216', 'MESH:D000077273', 'MESH:D000138', 'MESH:D000141', 'MESH:D000152', 'MESH:D000210', 'MESH:D000224', 'MESH:D000312', 'MESH:D000361', 'MESH:D000382', 'MESH:D000402', 'MESH:D000419', 'MESH:D000435', 'MESH:D000437', 'MESH:D000471', 'MESH:D000506', 'MESH:D000544', 'MESH:D000647', 'MESH:D000690', 'MESH:D000744', 'MESH:D000749', 'MESH:D000756', 'MESH:D000782', 'MESH:D000783', 'MESH:D000784', 'MESH:D000787', 'MESH:D000853', 'MESH:D000855', 'MESH:D000856', 'MESH:D000860', 'MESH:D001008', 'MESH:D001010', 'MESH:D001019', 'MESH:

,DiseaseID,DiseaseName,primekg_disease_node_id,primekg_disease_node_index,primekg_disease_node_name,disease_in_primekg
0,MESH:D000013,Congenital Abnormalities,839,35598,congenital abnormality,True
1,MESH:D000013,Congenital Abnormalities,839,35598,congenital abnormality,True
2,MESH:D000013,Congenital Abnormalities,839,35598,congenital abnormality,True
3,MESH:D000013,Congenital Abnormalities,839,35598,congenital abnormality,True
4,MESH:D000013,Congenital Abnormalities,839,35598,congenital abnormality,True



Unmatched disease row examples:


,DrugID,DrugName,DiseaseID,DiseaseName,DiseaseDescription
110,DB00755,Tretinoin,MESH:D000015,"Abnormalities, Multiple",Congenital abnormalities that affect more than...
137,DB00619,Imatinib,MESH:D000022,"Abortion, Spontaneous",Expulsion of the product of FERTILIZATION befo...
151,DB02709,Resveratrol,MESH:D000067877,Autism Spectrum Disorder,Wide continuum of associated cognitive and neu...
156,DB02709,Resveratrol,MESH:D000068079,Motor Disorders,Motor skills deficits that significantly and p...
165,DB11672,Curcumin,MESH:D000070642,"Brain Injuries, Traumatic",A form of acquired brain injury which occurs w...
167,DB01112,Cefuroxime,MESH:D000072660,Teratozoospermia,Conditions in which sperm show abnormal morpho...
170,DB00394,Beclomethasone dipropionate,MESH:D000075222,Essential Hypertension,"Hypertension that occurs without known cause, ..."
174,DB05294,Vandetanib,MESH:D000077192,Adenocarcinoma of Lung,A carcinoma originating in the lung and the mo...
188,DB00290,Bleomycin,MESH:D000077195,Squamous Cell Carcinoma of Head and Neck,The most common type of head and neck carcinom...
197,DB00997,Doxorubicin,MESH:D000077216,"Carcinoma, Ovarian Epithelial",A malignant neoplasm that originates in cells ...


In [7]:
# ============================================================
# 7. Initial graph-present / graph-absent split
# ============================================================

mirage["graph_present"] = mirage["drug_in_primekg"] & mirage["disease_in_primekg"]
mirage["graph_absent"] = ~mirage["graph_present"]

# More detailed absence type
def get_absence_type(row):
    if row["drug_in_primekg"] and row["disease_in_primekg"]:
        return "graph_present"
    if (not row["drug_in_primekg"]) and row["disease_in_primekg"]:
        return "drug_absent"
    if row["drug_in_primekg"] and (not row["disease_in_primekg"]):
        return "disease_absent"
    return "drug_and_disease_absent"

mirage["graph_absence_type"] = mirage.apply(get_absence_type, axis=1)

# ============================================================
# Summary
# ============================================================

n_total = len(mirage)
n_present = int(mirage["graph_present"].sum())
n_absent = int(mirage["graph_absent"].sum())

print("Initial graph-present / graph-absent split:")
print(f"Total MiRAGE pairs: {n_total:,}")
print(f"Graph-present pairs: {n_present:,} ({n_present / n_total:.4%})")
print(f"Graph-absent pairs: {n_absent:,} ({n_absent / n_total:.4%})")

print("\nAbsence type distribution:")
absence_counts = (
    mirage["graph_absence_type"]
    .value_counts()
    .rename_axis("graph_absence_type")
    .reset_index(name="count")
)
absence_counts["ratio"] = absence_counts["count"] / n_total
display(absence_counts)

print("\nDrug/Disease PrimeKG mapping crosstab:")
display(
    pd.crosstab(
        mirage["drug_in_primekg"],
        mirage["disease_in_primekg"],
        rownames=["drug_in_primekg"],
        colnames=["disease_in_primekg"],
        margins=True,
    )
)

# Unique entity-level summary
unique_drug_summary = mirage[["DrugID", "drug_in_primekg"]].drop_duplicates()
unique_disease_summary = mirage[["DiseaseID", "disease_in_primekg"]].drop_duplicates()

print("\nUnique entity-level mapping summary:")
print(
    f"Unique drugs in PrimeKG: {unique_drug_summary['drug_in_primekg'].sum():,} / "
    f"{len(unique_drug_summary):,} "
    f"({unique_drug_summary['drug_in_primekg'].mean():.4%})"
)
print(
    f"Unique diseases in PrimeKG: {unique_disease_summary['disease_in_primekg'].sum():,} / "
    f"{len(unique_disease_summary):,} "
    f"({unique_disease_summary['disease_in_primekg'].mean():.4%})"
)

print("\nGraph-present examples:")
display(
    mirage.loc[
        mirage["graph_present"],
        [
            "DrugID",
            "DrugName",
            "DiseaseID",
            "DiseaseName",
            "primekg_drug_node_index",
            "primekg_disease_node_index",
            "graph_absence_type",
        ],
    ].head()
)

print("\nGraph-absent examples:")
display(
    mirage.loc[
        mirage["graph_absent"],
        [
            "DrugID",
            "DrugName",
            "DiseaseID",
            "DiseaseName",
            "drug_in_primekg",
            "disease_in_primekg",
            "graph_absence_type",
        ],
    ].head()
)

Initial graph-present / graph-absent split:
Total MiRAGE pairs: 42,200
Graph-present pairs: 22,606 (53.5687%)
Graph-absent pairs: 19,594 (46.4313%)

Absence type distribution:


,graph_absence_type,count,ratio
0,graph_present,22606,0.535687
1,disease_absent,19592,0.464265
2,drug_and_disease_absent,1,0.000024
3,drug_absent,1,0.000024



Drug/Disease PrimeKG mapping crosstab:


disease_in_primekg,False,True,All
drug_in_primekg,,,
False,1,1,2
True,19592,22606,42198
All,19593,22607,42200



Unique entity-level mapping summary:
Unique drugs in PrimeKG: 1,408 / 1,410 (99.8582%)
Unique diseases in PrimeKG: 861 / 1,573 (54.7362%)

Graph-present examples:


,DrugID,DrugName,DiseaseID,DiseaseName,primekg_drug_node_index,primekg_disease_node_index,graph_absence_type
0,DB09140,Oxygen,MESH:D000013,Congenital Abnormalities,14013.0,35598,graph_present
1,DB00730,Thiabendazole,MESH:D000013,Congenital Abnormalities,15307.0,35598,graph_present
2,DB00898,Ethanol,MESH:D000013,Congenital Abnormalities,14587.0,35598,graph_present
3,DB01168,Procarbazine,MESH:D000013,Congenital Abnormalities,14914.0,35598,graph_present
4,DB00550,Propylthiouracil,MESH:D000013,Congenital Abnormalities,14640.0,35598,graph_present



Graph-absent examples:


,DrugID,DrugName,DiseaseID,DiseaseName,drug_in_primekg,disease_in_primekg,graph_absence_type
110,DB00755,Tretinoin,MESH:D000015,"Abnormalities, Multiple",True,False,disease_absent
111,DB00794,Primidone,MESH:D000015,"Abnormalities, Multiple",True,False,disease_absent
112,DB14001,alpha-Tocopherol succinate,MESH:D000015,"Abnormalities, Multiple",True,False,disease_absent
113,DB00586,Diclofenac,MESH:D000015,"Abnormalities, Multiple",True,False,disease_absent
114,DB01097,Leflunomide,MESH:D000015,"Abnormalities, Multiple",True,False,disease_absent


In [8]:
# ============================================================
# 8. Mark MiRAGE pairs already present in PrimeKG drug-disease relations
# ============================================================

def load_primekg_known_drug_disease_pairs():
    """
    Load known drug-disease pairs from PrimeKG.

    Returns
    -------
    known_pairs : set of (drug_node_id, disease_node_id)
        IDs are normalized PrimeKG node IDs.
    known_pair_relations : dict
        (drug_node_id, disease_node_id) -> set(relations)
    kg_source : str
        Which PrimeKG file was used.
    """
    known_pairs = set()
    known_pair_relations = {}

    # Prefer kg.csv if available because it usually has x_type/y_type/x_id/y_id.
    if PRIMEKG_KG_PATH.exists():
        kg_source = str(PRIMEKG_KG_PATH)
        kg = pd.read_csv(PRIMEKG_KG_PATH)

        required_cols = {"x_type", "x_id", "relation", "y_type", "y_id"}
        missing = required_cols - set(kg.columns)
        assert not missing, f"PrimeKG kg.csv missing columns: {missing}"

        kg["x_type"] = kg["x_type"].astype(str)
        kg["y_type"] = kg["y_type"].astype(str)
        kg["relation"] = kg["relation"].astype(str)
        kg["x_id"] = kg["x_id"].apply(normalize_primekg_node_id)
        kg["y_id"] = kg["y_id"].apply(normalize_primekg_node_id)

        dd = kg[kg["relation"].isin(PRIMEKG_DD_RELATIONS)].copy()

        for _, row in tqdm(dd.iterrows(), total=len(dd), desc="Collect PrimeKG known DD pairs"):
            x_type = row["x_type"]
            y_type = row["y_type"]
            rel = row["relation"]

            if x_type == "drug" and y_type == "disease":
                drug_id = normalize_primekg_node_id(row["x_id"])
                disease_id = normalize_primekg_node_id(row["y_id"])
            elif x_type == "disease" and y_type == "drug":
                drug_id = normalize_primekg_node_id(row["y_id"])
                disease_id = normalize_primekg_node_id(row["x_id"])
            else:
                continue

            if pd.isna(drug_id) or pd.isna(disease_id):
                continue

            pair = (drug_id, disease_id)
            known_pairs.add(pair)
            known_pair_relations.setdefault(pair, set()).add(rel)

        return known_pairs, known_pair_relations, kg_source

    # Fallback to edges.csv if kg.csv is unavailable.
    if PRIMEKG_EDGES_PATH.exists():
        kg_source = str(PRIMEKG_EDGES_PATH)
        edges = pd.read_csv(PRIMEKG_EDGES_PATH)

        required_cols = {"x_index", "y_index", "relation"}
        missing = required_cols - set(edges.columns)
        assert not missing, f"PrimeKG edges.csv missing columns: {missing}"

        nodes_by_index = (
            node_df
            .set_index("node_index")[["node_id", "node_type", "node_name"]]
            .to_dict("index")
        )

        edges["relation"] = edges["relation"].astype(str)
        dd = edges[edges["relation"].isin(PRIMEKG_DD_RELATIONS)].copy()

        for _, row in tqdm(dd.iterrows(), total=len(dd), desc="Collect PrimeKG known DD pairs"):
            x_idx = row["x_index"]
            y_idx = row["y_index"]
            rel = row["relation"]

            if x_idx not in nodes_by_index or y_idx not in nodes_by_index:
                continue

            x_node = nodes_by_index[x_idx]
            y_node = nodes_by_index[y_idx]

            if x_node["node_type"] == "drug" and y_node["node_type"] == "disease":
                drug_id = normalize_primekg_node_id(x_node["node_id"])
                disease_id = normalize_primekg_node_id(y_node["node_id"])
            elif x_node["node_type"] == "disease" and y_node["node_type"] == "drug":
                drug_id = normalize_primekg_node_id(y_node["node_id"])
                disease_id = normalize_primekg_node_id(x_node["node_id"])
            else:
                continue

            if pd.isna(drug_id) or pd.isna(disease_id):
                continue

            pair = (drug_id, disease_id)
            known_pairs.add(pair)
            known_pair_relations.setdefault(pair, set()).add(rel)

        return known_pairs, known_pair_relations, kg_source

    raise FileNotFoundError(
        f"Neither {PRIMEKG_KG_PATH} nor {PRIMEKG_EDGES_PATH} exists."
    )


primekg_known_dd_pairs, primekg_known_dd_pair_relations, primekg_dd_source = (
    load_primekg_known_drug_disease_pairs()
)

print(f"PrimeKG DD source: {primekg_dd_source}")
print(f"Known PrimeKG drug-disease pairs in selected relations: {len(primekg_known_dd_pairs):,}")


def mirage_pair_in_primekg_known_dd(row):
    """
    Only graph-present rows can overlap PrimeKG by node ID.
    Graph-absent rows are marked False because at least one node is absent.
    """
    if not row["graph_present"]:
        return False

    pair = (
        normalize_primekg_node_id(row["primekg_drug_node_id"]),
        normalize_primekg_node_id(row["primekg_disease_node_id"]),
    )
    return pair in primekg_known_dd_pairs


def mirage_primekg_known_dd_relations(row):
    if not row["graph_present"]:
        return ""

    pair = (
        normalize_primekg_node_id(row["primekg_drug_node_id"]),
        normalize_primekg_node_id(row["primekg_disease_node_id"]),
    )

    return "|".join(sorted(primekg_known_dd_pair_relations.get(pair, [])))


mirage["pair_in_primekg_known_dd"] = mirage.apply(mirage_pair_in_primekg_known_dd, axis=1)
mirage["primekg_known_dd_relations"] = mirage.apply(mirage_primekg_known_dd_relations, axis=1)

# ============================================================
# Summary
# ============================================================

n_primekg_overlap = int(mirage["pair_in_primekg_known_dd"].sum())
n_graph_present = int(mirage["graph_present"].sum())

print("\nMiRAGE overlap with PrimeKG known DD relations:")
print(f"Overlapping rows: {n_primekg_overlap:,} / {len(mirage):,} ({n_primekg_overlap / len(mirage):.4%})")
print(f"Overlapping among graph-present rows: {n_primekg_overlap:,} / {n_graph_present:,} ({n_primekg_overlap / n_graph_present:.4%})")

print("\nOverlap relation distribution:")
display(
    mirage.loc[mirage["pair_in_primekg_known_dd"], "primekg_known_dd_relations"]
    .value_counts()
    .rename_axis("primekg_known_dd_relations")
    .reset_index(name="count")
)

print("\nPrimeKG-overlap examples:")
display(
    mirage.loc[
        mirage["pair_in_primekg_known_dd"],
        [
            "DrugID",
            "DrugName",
            "DiseaseID",
            "DiseaseName",
            "primekg_drug_node_id",
            "primekg_disease_node_id",
            "primekg_known_dd_relations",
        ],
    ].head(20)
)

Collect PrimeKG known DD pairs:   0%|          | 0/85262 [00:00<?, ?it/s]

PrimeKG DD source: ..\data\kg\kg.csv
Known PrimeKG drug-disease pairs in selected relations: 42,383

MiRAGE overlap with PrimeKG known DD relations:
Overlapping rows: 2,678 / 42,200 (6.3460%)
Overlapping among graph-present rows: 2,678 / 22,606 (11.8464%)

Overlap relation distribution:


,primekg_known_dd_relations,count
0,contraindication,1638
1,indication,803
2,off-label use,209
3,contraindication|indication,14
4,indication|off-label use,10
5,contraindication|off-label use,4



PrimeKG-overlap examples:


,DrugID,DrugName,DiseaseID,DiseaseName,primekg_drug_node_id,primekg_disease_node_id,primekg_known_dd_relations
271,DB01016,Glyburide,MESH:D000140,"Acidosis, Lactic",DB01016,6040,contraindication
273,DB00495,Zidovudine,MESH:D000140,"Acidosis, Lactic",DB00495,6040,contraindication
278,DB00625,Efavirenz,MESH:D000140,"Acidosis, Lactic",DB00625,6040,contraindication
285,DB00709,Lamivudine,MESH:D000140,"Acidosis, Lactic",DB00709,6040,contraindication
286,DB00649,Stavudine,MESH:D000140,"Acidosis, Lactic",DB00649,6040,contraindication
290,DB00900,Didanosine,MESH:D000140,"Acidosis, Lactic",DB00900,6040,contraindication
292,DB00331,Metformin,MESH:D000140,"Acidosis, Lactic",DB00331,6040,contraindication
295,DB00300,Tenofovir disoproxil,MESH:D000140,"Acidosis, Lactic",DB00300,6040,contraindication
345,DB00503,Ritonavir,MESH:D000163,Acquired Immunodeficiency Syndrome,DB00503,12268,indication
346,DB00943,Zalcitabine,MESH:D000163,Acquired Immunodeficiency Syndrome,DB00943,12268,indication


In [9]:
# ============================================================
# 9. Mark MiRAGE pairs overlapping raw LLM training sources:
#    RepoDB and DrugRepoBank
# ============================================================

def add_train_source_pair(pair_dict, drug_id, disease_name, source_name):
    """
    Add a raw train-source pair using:
        key = (DrugBank ID, normalized disease name)
    """
    drug_id = canonical_drugbank_id(drug_id)
    disease_norm = normalize_text(disease_name)

    if pd.isna(drug_id) or disease_norm == "":
        return

    key = (drug_id, disease_norm)
    pair_dict.setdefault(key, set()).add(source_name)


train_source_name_pair_sources = {}

# ------------------------------------------------------------
# 9.1 RepoDB pairs
# ------------------------------------------------------------
# RepoDB example:
# "drug_name","drugbank_id","ind_name","ind_id","NCT","status","phase","DetailedStatus"
#
# For overlap filtering, use drugbank_id + ind_name.
# We do not filter by status here, because the LLM training material was generated
# from RepoDB drug-disease pairs, and any exact pair overlap should be removed
# from MiRAGE external test.
# ------------------------------------------------------------

repodb_pairs_before = len(train_source_name_pair_sources)

for _, row in repodb_df.iterrows():
    add_train_source_pair(
        train_source_name_pair_sources,
        drug_id=row.get("drugbank_id", pd.NA),
        disease_name=row.get("ind_name", pd.NA),
        source_name="RepoDB:ind_name",
    )

repodb_pairs_added = len(train_source_name_pair_sources) - repodb_pairs_before

print("RepoDB raw train-source pairs loaded:")
print(f"Unique name-level pairs after RepoDB: {len(train_source_name_pair_sources):,}")
print(f"New unique pairs added by RepoDB: {repodb_pairs_added:,}")


# ------------------------------------------------------------
# 9.2 DrugRepoBank pairs
# ------------------------------------------------------------
# DrugRepoBank columns include:
#   DrugID, Disease, NewDisease
#
# For drug repositioning, NewDisease is usually the target/new disease.
# However, because the training material may have used the full row context,
# we conservatively include both:
#   DrugID + NewDisease
#   DrugID + Disease
#
# Later the output keeps source tags so you can inspect how many overlaps come
# from each column.
# ------------------------------------------------------------

drb_pairs_before = len(train_source_name_pair_sources)

for _, row in drugrepobank_df.iterrows():
    drug_id = row.get("DrugID", pd.NA)

    if "NewDisease" in drugrepobank_df.columns:
        add_train_source_pair(
            train_source_name_pair_sources,
            drug_id=drug_id,
            disease_name=row.get("NewDisease", pd.NA),
            source_name="DrugRepoBank:NewDisease",
        )

    if "Disease" in drugrepobank_df.columns:
        add_train_source_pair(
            train_source_name_pair_sources,
            drug_id=drug_id,
            disease_name=row.get("Disease", pd.NA),
            source_name="DrugRepoBank:Disease",
        )

drb_pairs_added = len(train_source_name_pair_sources) - drb_pairs_before

print("\nDrugRepoBank raw train-source pairs loaded:")
print(f"Unique name-level pairs after DrugRepoBank: {len(train_source_name_pair_sources):,}")
print(f"New unique pairs added by DrugRepoBank: {drb_pairs_added:,}")


# ------------------------------------------------------------
# 9.3 Mark MiRAGE overlaps
# ------------------------------------------------------------

def get_train_source_overlap_sources(row):
    key = (
        canonical_drugbank_id(row["DrugID"]),
        normalize_text(row["DiseaseName"]),
    )

    sources = train_source_name_pair_sources.get(key, set())
    return "|".join(sorted(sources))


mirage["train_source_overlap_sources"] = mirage.apply(
    get_train_source_overlap_sources,
    axis=1,
)

mirage["pair_in_llm_train_source_raw"] = mirage["train_source_overlap_sources"].astype(str).str.len() > 0


# ============================================================
# Summary
# ============================================================

n_train_overlap = int(mirage["pair_in_llm_train_source_raw"].sum())

print("\nMiRAGE overlap with raw LLM training sources:")
print(
    f"Overlapping rows: {n_train_overlap:,} / {len(mirage):,} "
    f"({n_train_overlap / len(mirage):.4%})"
)

print("\nOverlap source distribution:")
display(
    mirage.loc[mirage["pair_in_llm_train_source_raw"], "train_source_overlap_sources"]
    .value_counts()
    .rename_axis("train_source_overlap_sources")
    .reset_index(name="count")
)

print("\nOverlap by graph-present / graph-absent:")
display(
    pd.crosstab(
        mirage["graph_absence_type"],
        mirage["pair_in_llm_train_source_raw"],
        rownames=["graph_absence_type"],
        colnames=["pair_in_llm_train_source_raw"],
        margins=True,
    )
)

print("\nRaw train-source overlap examples:")
display(
    mirage.loc[
        mirage["pair_in_llm_train_source_raw"],
        [
            "DrugID",
            "DrugName",
            "DiseaseID",
            "DiseaseName",
            "graph_absence_type",
            "train_source_overlap_sources",
        ],
    ].head(30)
)

RepoDB raw train-source pairs loaded:
Unique name-level pairs after RepoDB: 10,602
New unique pairs added by RepoDB: 10,602

DrugRepoBank raw train-source pairs loaded:
Unique name-level pairs after DrugRepoBank: 10,895
New unique pairs added by DrugRepoBank: 293

MiRAGE overlap with raw LLM training sources:
Overlapping rows: 958 / 42,200 (2.2701%)

Overlap source distribution:


,train_source_overlap_sources,count
0,RepoDB:ind_name,945
1,DrugRepoBank:Disease|RepoDB:ind_name,7
2,DrugRepoBank:Disease,6



Overlap by graph-present / graph-absent:


pair_in_llm_train_source_raw,False,True,All
graph_absence_type,,,
disease_absent,19194,398,19592
drug_absent,1,0,1
drug_and_disease_absent,1,0,1
graph_present,22046,560,22606
All,41242,958,42200



Raw train-source overlap examples:


,DrugID,DrugName,DiseaseID,DiseaseName,graph_absence_type,train_source_overlap_sources
318,DB00977,Ethinylestradiol,MESH:D000152,Acne Vulgaris,disease_absent,RepoDB:ind_name
322,DB00210,Adapalene,MESH:D000152,Acne Vulgaris,disease_absent,RepoDB:ind_name
324,DB00755,Tretinoin,MESH:D000152,Acne Vulgaris,disease_absent,RepoDB:ind_name
328,DB01190,Clindamycin,MESH:D000152,Acne Vulgaris,disease_absent,RepoDB:ind_name
329,DB00548,Azelaic acid,MESH:D000152,Acne Vulgaris,disease_absent,RepoDB:ind_name
331,DB00250,Dapsone,MESH:D000152,Acne Vulgaris,disease_absent,RepoDB:ind_name
333,DB00759,Tetracycline,MESH:D000152,Acne Vulgaris,disease_absent,RepoDB:ind_name
337,DB00199,Erythromycin,MESH:D000152,Acne Vulgaris,disease_absent,RepoDB:ind_name
339,DB09096,Benzoyl peroxide,MESH:D000152,Acne Vulgaris,disease_absent,RepoDB:ind_name
340,DB00717,Norethisterone,MESH:D000152,Acne Vulgaris,disease_absent,RepoDB:ind_name


In [10]:
# ============================================================
# 10. Create final unseen external subsets
# ============================================================

mirage["exclude_primekg_overlap"] = mirage["pair_in_primekg_known_dd"]
mirage["exclude_llm_train_source_overlap"] = mirage["pair_in_llm_train_source_raw"]

# Final exclusion flag:
# remove any MiRAGE pair already present in shared PrimeKG DD relations
# or in raw LLM training sources.
mirage["exclude_any_overlap"] = (
    mirage["exclude_primekg_overlap"]
    | mirage["exclude_llm_train_source_overlap"]
)

# Final unseen external benchmark
mirage_unseen = mirage.loc[~mirage["exclude_any_overlap"]].copy().reset_index(drop=True)

# Split unseen benchmark into graph-present and graph-absent subsets
mirage_graph_present_unseen = (
    mirage_unseen.loc[mirage_unseen["graph_present"]]
    .copy()
    .reset_index(drop=True)
)

mirage_graph_absent_unseen = (
    mirage_unseen.loc[mirage_unseen["graph_absent"]]
    .copy()
    .reset_index(drop=True)
)

# Excluded subsets for audit
mirage_excluded_primekg_overlap = (
    mirage.loc[mirage["exclude_primekg_overlap"]]
    .copy()
    .reset_index(drop=True)
)

mirage_excluded_llm_train_source_overlap = (
    mirage.loc[mirage["exclude_llm_train_source_overlap"]]
    .copy()
    .reset_index(drop=True)
)

mirage_excluded_any_overlap = (
    mirage.loc[mirage["exclude_any_overlap"]]
    .copy()
    .reset_index(drop=True)
)

# ============================================================
# Summary
# ============================================================

n_total = len(mirage)
n_primekg = int(mirage["exclude_primekg_overlap"].sum())
n_llm_train = int(mirage["exclude_llm_train_source_overlap"].sum())
n_any = int(mirage["exclude_any_overlap"].sum())

print("Final MiRAGE external split summary:")
print(f"Original MiRAGE pairs: {n_total:,}")
print(f"Excluded due to PrimeKG known DD overlap: {n_primekg:,} ({n_primekg / n_total:.4%})")
print(f"Excluded due to raw LLM train-source overlap: {n_llm_train:,} ({n_llm_train / n_total:.4%})")
print(f"Excluded due to any overlap union: {n_any:,} ({n_any / n_total:.4%})")
print(f"Remaining unseen MiRAGE pairs: {len(mirage_unseen):,} ({len(mirage_unseen) / n_total:.4%})")

print("\nFinal unseen subsets:")
print(
    f"Graph-present unseen: {len(mirage_graph_present_unseen):,} "
    f"({len(mirage_graph_present_unseen) / n_total:.4%} of original; "
    f"{len(mirage_graph_present_unseen) / len(mirage_unseen):.4%} of unseen)"
)
print(
    f"Graph-absent unseen: {len(mirage_graph_absent_unseen):,} "
    f"({len(mirage_graph_absent_unseen) / n_total:.4%} of original; "
    f"{len(mirage_graph_absent_unseen) / len(mirage_unseen):.4%} of unseen)"
)

print("\nOverlap reason crosstab:")
display(
    pd.crosstab(
        mirage["exclude_primekg_overlap"],
        mirage["exclude_llm_train_source_overlap"],
        rownames=["exclude_primekg_overlap"],
        colnames=["exclude_llm_train_source_overlap"],
        margins=True,
    )
)

print("\nFinal subset by graph absence type:")
final_subset_summary = (
    mirage_unseen["graph_absence_type"]
    .value_counts()
    .rename_axis("graph_absence_type")
    .reset_index(name="count")
)
final_subset_summary["ratio_of_unseen"] = final_subset_summary["count"] / len(mirage_unseen)
final_subset_summary["ratio_of_original"] = final_subset_summary["count"] / n_total
display(final_subset_summary)

print("\nExcluded rows by graph absence type:")
excluded_subset_summary = (
    mirage_excluded_any_overlap["graph_absence_type"]
    .value_counts()
    .rename_axis("graph_absence_type")
    .reset_index(name="count")
)
excluded_subset_summary["ratio_of_excluded"] = excluded_subset_summary["count"] / len(mirage_excluded_any_overlap)
excluded_subset_summary["ratio_of_original"] = excluded_subset_summary["count"] / n_total
display(excluded_subset_summary)

print("\nFinal graph-present unseen examples:")
display(
    mirage_graph_present_unseen[
        [
            "DrugID",
            "DrugName",
            "DiseaseID",
            "DiseaseName",
            "primekg_drug_node_id",
            "primekg_disease_node_id",
            "graph_absence_type",
            "exclude_primekg_overlap",
            "exclude_llm_train_source_overlap",
        ]
    ].head(10)
)

print("\nFinal graph-absent unseen examples:")
display(
    mirage_graph_absent_unseen[
        [
            "DrugID",
            "DrugName",
            "DiseaseID",
            "DiseaseName",
            "drug_in_primekg",
            "disease_in_primekg",
            "graph_absence_type",
            "exclude_primekg_overlap",
            "exclude_llm_train_source_overlap",
        ]
    ].head(10)
)

# ============================================================
# Sanity checks
# ============================================================

assert len(mirage_unseen) + len(mirage_excluded_any_overlap) == len(mirage)
assert not mirage_unseen["exclude_any_overlap"].any()
assert not mirage_graph_present_unseen["exclude_any_overlap"].any()
assert not mirage_graph_absent_unseen["exclude_any_overlap"].any()
assert mirage_graph_present_unseen["graph_present"].all()
assert mirage_graph_absent_unseen["graph_absent"].all()

print("\nSanity checks passed.")

Final MiRAGE external split summary:
Original MiRAGE pairs: 42,200
Excluded due to PrimeKG known DD overlap: 2,678 (6.3460%)
Excluded due to raw LLM train-source overlap: 958 (2.2701%)
Excluded due to any overlap union: 3,214 (7.6161%)
Remaining unseen MiRAGE pairs: 38,986 (92.3839%)

Final unseen subsets:
Graph-present unseen: 19,790 (46.8957% of original; 50.7618% of unseen)
Graph-absent unseen: 19,196 (45.4882% of original; 49.2382% of unseen)

Overlap reason crosstab:


exclude_llm_train_source_overlap,False,True,All
exclude_primekg_overlap,,,
False,38986,536,39522
True,2256,422,2678
All,41242,958,42200



Final subset by graph absence type:


,graph_absence_type,count,ratio_of_unseen,ratio_of_original
0,graph_present,19790,0.507618,0.468957
1,disease_absent,19194,0.492331,0.454834
2,drug_and_disease_absent,1,0.000026,0.000024
3,drug_absent,1,0.000026,0.000024



Excluded rows by graph absence type:


...


Final graph-present unseen examples:


,DrugID,DrugName,DiseaseID,DiseaseName,primekg_drug_node_id,primekg_disease_node_id,graph_absence_type,exclude_primekg_overlap,exclude_llm_train_source_overlap
0,DB09140,Oxygen,MESH:D000013,Congenital Abnormalities,DB09140,839,graph_present,False,False
1,DB00730,Thiabendazole,MESH:D000013,Congenital Abnormalities,DB00730,839,graph_present,False,False
2,DB00898,Ethanol,MESH:D000013,Congenital Abnormalities,DB00898,839,graph_present,False,False
3,DB01168,Procarbazine,MESH:D000013,Congenital Abnormalities,DB01168,839,graph_present,False,False
4,DB00550,Propylthiouracil,MESH:D000013,Congenital Abnormalities,DB00550,839,graph_present,False,False
5,DB00668,Epinephrine,MESH:D000013,Congenital Abnormalities,DB00668,839,graph_present,False,False
6,DB00307,Bexarotene,MESH:D000013,Congenital Abnormalities,DB00307,839,graph_present,False,False
7,DB00313,Valproic acid,MESH:D000013,Congenital Abnormalities,DB00313,839,graph_present,False,False
8,DB01181,Ifosfamide,MESH:D000013,Congenital Abnormalities,DB01181,839,graph_present,False,False
9,DB00347,Trimethadione,MESH:D000013,Congenital Abnormalities,DB00347,839,graph_present,False,False



Final graph-absent unseen examples:


,DrugID,DrugName,DiseaseID,DiseaseName,drug_in_primekg,disease_in_primekg,graph_absence_type,exclude_primekg_overlap,exclude_llm_train_source_overlap
0,DB00755,Tretinoin,MESH:D000015,"Abnormalities, Multiple",True,False,disease_absent,False,False
1,DB00794,Primidone,MESH:D000015,"Abnormalities, Multiple",True,False,disease_absent,False,False
2,DB14001,alpha-Tocopherol succinate,MESH:D000015,"Abnormalities, Multiple",True,False,disease_absent,False,False
3,DB00586,Diclofenac,MESH:D000015,"Abnormalities, Multiple",True,False,disease_absent,False,False
4,DB01097,Leflunomide,MESH:D000015,"Abnormalities, Multiple",True,False,disease_absent,False,False
5,DB11155,Triclocarban,MESH:D000015,"Abnormalities, Multiple",True,False,disease_absent,False,False
6,DB00929,Misoprostol,MESH:D000015,"Abnormalities, Multiple",True,False,disease_absent,False,False
7,DB00531,Cyclophosphamide,MESH:D000015,"Abnormalities, Multiple",True,False,disease_absent,False,False
8,DB00347,Trimethadione,MESH:D000015,"Abnormalities, Multiple",True,False,disease_absent,False,False
9,DB00480,Lenalidomide,MESH:D000015,"Abnormalities, Multiple",True,False,disease_absent,False,False



Sanity checks passed.


In [11]:
# ============================================================
# 11. Prepare final MiRAGE external split persistence helpers
# ============================================================

FINAL_SUMMARY_PATH = OUTPUT_DIR / "summary.json"
FINAL_GRAPH_PRESENT_PATH = OUTPUT_DIR / "mirage_graph_present.csv"
FINAL_GRAPH_UNPRESENT_PATH = OUTPUT_DIR / "mirage_graph_unpresent.csv"

LEGACY_VERBOSE_OUTPUT_PATHS = [
    OUTPUT_DIR / "mirage_all_mapped_with_flags.csv",
    OUTPUT_DIR / "mirage_unseen.csv",
    OUTPUT_DIR / "mirage_graph_present_unseen.csv",
    OUTPUT_DIR / "mirage_graph_absent_unseen.csv",
    OUTPUT_DIR / "mirage_excluded_primekg_overlap.csv",
    OUTPUT_DIR / "mirage_excluded_llm_train_source_overlap.csv",
    OUTPUT_DIR / "mirage_excluded_any_overlap.csv",
    OUTPUT_DIR / "mirage_external_split_summary.json",
    OUTPUT_DIR / "mirage_graph_present_unseen_with_txgnn_idx.csv",
    OUTPUT_DIR / "txgnn_external_test_positive_edges_raw.csv",
    OUTPUT_DIR / "txgnn_external_test_positive_edges.csv",
    OUTPUT_DIR / "txgnn_external_test_duplicate_edges.csv",
]


def safe_ratio(numerator, denominator):
    return float(numerator / denominator) if denominator else 0.0


def remove_legacy_mirage_output_files():
    removed_paths = []
    for legacy_path in LEGACY_VERBOSE_OUTPUT_PATHS:
        if legacy_path.exists():
            legacy_path.unlink()
            removed_paths.append(legacy_path)
    return removed_paths


def build_mirage_external_split_summary(graph_present_to_save):
    unique_drugs_total = int(mirage[["DrugID"]].drop_duplicates().shape[0])
    unique_drugs_in_primekg = int(
        mirage[["DrugID", "drug_in_primekg"]]
        .drop_duplicates()["drug_in_primekg"]
        .sum()
    )
    unique_diseases_total = int(mirage[["DiseaseID"]].drop_duplicates().shape[0])
    unique_diseases_in_primekg = int(
        mirage[["DiseaseID", "disease_in_primekg"]]
        .drop_duplicates()["disease_in_primekg"]
        .sum()
    )

    summary = {
        "original_mirage_pairs": int(len(mirage)),
        "initial_graph_present_pairs": int(mirage["graph_present"].sum()),
        "initial_graph_unpresent_pairs": int(mirage["graph_absent"].sum()),
        "initial_graph_present_ratio": float(mirage["graph_present"].mean()),
        "initial_graph_unpresent_ratio": float(mirage["graph_absent"].mean()),
        "entity_coverage": {
            "unique_drugs_total": unique_drugs_total,
            "unique_drugs_in_primekg": unique_drugs_in_primekg,
            "unique_drugs_in_primekg_ratio": safe_ratio(unique_drugs_in_primekg, unique_drugs_total),
            "unique_diseases_total": unique_diseases_total,
            "unique_diseases_in_primekg": unique_diseases_in_primekg,
            "unique_diseases_in_primekg_ratio": safe_ratio(unique_diseases_in_primekg, unique_diseases_total),
        },
        "exclusion_counts": {
            "primekg_overlap": int(mirage["exclude_primekg_overlap"].sum()),
            "llm_train_source_overlap": int(mirage["exclude_llm_train_source_overlap"].sum()),
            "any_overlap": int(mirage["exclude_any_overlap"].sum()),
        },
        "exclusion_ratios_of_original": {
            "primekg_overlap": safe_ratio(int(mirage["exclude_primekg_overlap"].sum()), len(mirage)),
            "llm_train_source_overlap": safe_ratio(int(mirage["exclude_llm_train_source_overlap"].sum()), len(mirage)),
            "any_overlap": safe_ratio(int(mirage["exclude_any_overlap"].sum()), len(mirage)),
        },
        "remaining_unseen_pairs": int(len(mirage_unseen)),
        "remaining_unseen_ratio_of_original": safe_ratio(len(mirage_unseen), len(mirage)),
        "final_graph_present_pairs": int(len(graph_present_to_save)),
        "final_graph_unpresent_pairs": int(len(mirage_graph_absent_unseen)),
        "final_graph_present_ratio_of_original": safe_ratio(len(graph_present_to_save), len(mirage)),
        "final_graph_unpresent_ratio_of_original": safe_ratio(len(mirage_graph_absent_unseen), len(mirage)),
        "final_graph_present_ratio_of_unseen": safe_ratio(len(graph_present_to_save), len(mirage_unseen)),
        "final_graph_unpresent_ratio_of_unseen": safe_ratio(len(mirage_graph_absent_unseen), len(mirage_unseen)),
        "relation_and_filtering_definitions": {
            "primekg_dd_relations_used_for_overlap": sorted(list(PRIMEKG_DD_RELATIONS)),
            "default_external_relation_for_positive_edges": DEFAULT_EXTERNAL_RELATION,
            "use_raw_train_source_overlap_filter": bool(USE_RAW_TRAIN_SOURCE_OVERLAP_FILTER),
            "llm_train_source_overlap_matching": "DrugBank ID exact match + normalized disease name exact match",
            "negative_edges_sampled": False,
        },
        "source_paths": {
            "mirage_mapping_path": str(MIRAGE_MAPPING_PATH),
            "mirage_druginfo_path": str(MIRAGE_DRUGINFO_PATH),
            "mirage_diseaseinfo_path": str(MIRAGE_DISEASEINFO_PATH),
            "mondo_json_path": str(MONDO_JSON_PATH),
            "primekg_nodes_path": str(PRIMEKG_NODES_PATH),
            "primekg_kg_path": str(PRIMEKG_KG_PATH),
            "primekg_kg_directed_path": str(PRIMEKG_KG_DIRECTED_PATH),
            "repodb_raw_path": str(REPODB_RAW_PATH),
            "drugrepobank_raw_path": str(DRUGREPOBANK_RAW_PATH),
        },
        "saved_outputs": {
            "summary_json": str(FINAL_SUMMARY_PATH),
            "mirage_graph_present_csv": str(FINAL_GRAPH_PRESENT_PATH),
            "mirage_graph_unpresent_csv": str(FINAL_GRAPH_UNPRESENT_PATH),
        },
        "notes": [
            "No negative edges are sampled.",
            "mirage_graph_present.csv stores the final de-leaked graph-present MiRAGE set.",
            "mirage_graph_unpresent.csv stores the final de-leaked graph-absent/unpresent MiRAGE set.",
        ],
    }

    return summary


Saved MiRAGE external split outputs:
- All mapped with flags:          ..\data\processed\mirage_all_mapped_with_flags.csv
- Unseen external set:            ..\data\processed\mirage_unseen.csv
- Graph-present unseen set:       ..\data\processed\mirage_graph_present_unseen.csv
- Graph-absent unseen set:        ..\data\processed\mirage_graph_absent_unseen.csv
- Excluded PrimeKG overlap:       ..\data\processed\mirage_excluded_primekg_overlap.csv
- Excluded LLM train overlap:     ..\data\processed\mirage_excluded_llm_train_source_overlap.csv
- Excluded any overlap:           ..\data\processed\mirage_excluded_any_overlap.csv
- Summary JSON:                   ..\data\processed\mirage_external_split_summary.json

Summary:
{
  "original_mirage_pairs": 42200,
  "initial_graph_present_pairs": 22606,
  "initial_graph_absent_pairs": 19594,
  "initial_graph_present_ratio": 0.5356872037914692,
  "initial_graph_absent_ratio": 0.46431279620853083,
  "unique_drugs_total": 1410,
  "unique_drugs_in_prime

In [12]:
# ============================================================
# 12. Build TxGNN local node-index maps from kg_directed.csv
# ============================================================

assert PRIMEKG_KG_DIRECTED_PATH.exists(), (
    f"kg_directed.csv not found: {PRIMEKG_KG_DIRECTED_PATH}\n"
    "Please generate it using the TxGNN/PrimeKG preprocessing pipeline first."
)

kg_directed, kg_directed_enc = safe_read_csv(PRIMEKG_KG_DIRECTED_PATH)

print(f"Loaded kg_directed.csv: encoding={kg_directed_enc}, shape={kg_directed.shape}")
print("kg_directed columns:")
print(kg_directed.columns.tolist())

required_kg_directed_cols = {"x_type", "x_id", "x_idx", "relation", "y_type", "y_id", "y_idx"}
missing_kg_directed_cols = required_kg_directed_cols - set(kg_directed.columns)
assert not missing_kg_directed_cols, (
    f"kg_directed.csv missing columns: {missing_kg_directed_cols}"
)

# Normalize IDs
kg_directed["x_type"] = kg_directed["x_type"].astype(str)
kg_directed["y_type"] = kg_directed["y_type"].astype(str)
kg_directed["relation"] = kg_directed["relation"].astype(str)
kg_directed["x_id"] = kg_directed["x_id"].apply(normalize_primekg_node_id)
kg_directed["y_id"] = kg_directed["y_id"].apply(normalize_primekg_node_id)

# ============================================================
# Build type-specific local index maps
# ============================================================

def collect_local_idx_map(df, node_type):
    """
    Collect node_id -> local_idx for one node type from both x-side and y-side.

    In a DGL heterograph, local node indices are type-specific.
    """
    pieces = []

    x_part = (
        df.loc[df["x_type"] == node_type, ["x_id", "x_idx"]]
        .rename(columns={"x_id": "node_id", "x_idx": "local_idx"})
    )
    pieces.append(x_part)

    y_part = (
        df.loc[df["y_type"] == node_type, ["y_id", "y_idx"]]
        .rename(columns={"y_id": "node_id", "y_idx": "local_idx"})
    )
    pieces.append(y_part)

    out = pd.concat(pieces, ignore_index=True)
    out = out.dropna(subset=["node_id", "local_idx"]).copy()

    out["node_id"] = out["node_id"].apply(normalize_primekg_node_id)
    out["local_idx"] = out["local_idx"].astype(int)

    # Check whether a node_id maps to multiple local indices.
    conflict = (
        out.drop_duplicates()
        .groupby("node_id")["local_idx"]
        .nunique()
        .reset_index(name="n_local_idx")
    )
    conflict = conflict[conflict["n_local_idx"] > 1]

    if len(conflict) > 0:
        print(f"[WARN] {node_type}: found node_id with multiple local_idx.")
        display(conflict.head(20))

    # Keep first after dropping exact duplicates
    out = out.drop_duplicates(subset=["node_id", "local_idx"])
    out = out.drop_duplicates(subset=["node_id"], keep="first")

    return dict(zip(out["node_id"], out["local_idx"])), out


txgnn_drug_id_to_local_idx, txgnn_drug_local_df = collect_local_idx_map(kg_directed, "drug")
txgnn_disease_id_to_local_idx, txgnn_disease_local_df = collect_local_idx_map(kg_directed, "disease")

print("\nTxGNN local index map sizes:")
print(f"Drug local map size: {len(txgnn_drug_id_to_local_idx):,}")
print(f"Disease local map size: {len(txgnn_disease_id_to_local_idx):,}")

print("\nDrug local index examples:")
display(txgnn_drug_local_df.head())

print("\nDisease local index examples:")
display(txgnn_disease_local_df.head())

# ============================================================
# Check graph-present unseen coverage in kg_directed local maps
# ============================================================

tmp_gp = mirage_graph_present_unseen.copy()

tmp_gp["txgnn_drug_idx"] = tmp_gp["primekg_drug_node_id"].apply(
    lambda x: txgnn_drug_id_to_local_idx.get(normalize_primekg_node_id(x), pd.NA)
)
tmp_gp["txgnn_disease_idx"] = tmp_gp["primekg_disease_node_id"].apply(
    lambda x: txgnn_disease_id_to_local_idx.get(normalize_primekg_node_id(x), pd.NA)
)

mirage_graph_present_unseen_with_txgnn_idx = tmp_gp.copy()

missing_txgnn_drug_idx = tmp_gp["txgnn_drug_idx"].isna().sum()
missing_txgnn_disease_idx = tmp_gp["txgnn_disease_idx"].isna().sum()
missing_any_txgnn_idx = (
    tmp_gp["txgnn_drug_idx"].isna() | tmp_gp["txgnn_disease_idx"].isna()
).sum()

print("\nCoverage of graph-present unseen pairs in TxGNN local maps:")
print(f"Graph-present unseen rows: {len(tmp_gp):,}")
print(f"Missing TxGNN drug local idx rows: {missing_txgnn_drug_idx:,}")
print(f"Missing TxGNN disease local idx rows: {missing_txgnn_disease_idx:,}")
print(f"Missing either local idx rows: {missing_any_txgnn_idx:,}")

if missing_any_txgnn_idx > 0:
    print("\nExamples with missing TxGNN local idx:")
    display(
        tmp_gp.loc[
            tmp_gp["txgnn_drug_idx"].isna() | tmp_gp["txgnn_disease_idx"].isna(),
            [
                "DrugID",
                "DrugName",
                "DiseaseID",
                "DiseaseName",
                "primekg_drug_node_id",
                "primekg_disease_node_id",
                "txgnn_drug_idx",
                "txgnn_disease_idx",
            ],
        ].head(20)
    )
else:
    print("All graph-present unseen pairs have TxGNN local indices.")

Loaded kg_directed.csv: encoding=utf-8, shape=(4050249, 7)
kg_directed columns:
['x_type', 'x_id', 'relation', 'y_type', 'y_id', 'x_idx', 'y_idx']

TxGNN local index map sizes:
Drug local map size: 7,957
Disease local map size: 17,080

Drug local index examples:


,node_id,local_idx
0,DB09130,5810
2,DB09140,5819
3,DB00180,168
4,DB00240,228
5,DB00253,241



Disease local index examples:


,node_id,local_idx
0,13924_12592_14672_13460_12591_12536_30861_8146...,2502
2,11160_13119_13978_12060_12327_12670_13210_1106...,1038
5,8099_12497_12498,15420
6,14854_14293_14470_12380_11832_14603_14853_1176...,2962
7,33202_32776_30905_33670_33200_32740_32732_3320...,10457



Coverage of graph-present unseen pairs in TxGNN local maps:
Graph-present unseen rows: 19,790
Missing TxGNN drug local idx rows: 0
Missing TxGNN disease local idx rows: 0
Missing either local idx rows: 0
All graph-present unseen pairs have TxGNN local indices.


In [15]:
# ============================================================
# 13. Create TxGNN/GNN external positive edge table
# ============================================================

txgnn_graph_present = mirage_graph_present_unseen.copy()

# Map PrimeKG node IDs to TxGNN type-specific local indices
txgnn_graph_present["txgnn_drug_idx"] = txgnn_graph_present["primekg_drug_node_id"].apply(
    lambda x: txgnn_drug_id_to_local_idx.get(normalize_primekg_node_id(x), pd.NA)
)
txgnn_graph_present["txgnn_disease_idx"] = txgnn_graph_present["primekg_disease_node_id"].apply(
    lambda x: txgnn_disease_id_to_local_idx.get(normalize_primekg_node_id(x), pd.NA)
)

# Keep only rows evaluable by TxGNN local graph
txgnn_graph_present_evaluable = (
    txgnn_graph_present
    .dropna(subset=["txgnn_drug_idx", "txgnn_disease_idx"])
    .copy()
    .reset_index(drop=True)
)

txgnn_graph_present_evaluable["txgnn_drug_idx"] = txgnn_graph_present_evaluable["txgnn_drug_idx"].astype(int)
txgnn_graph_present_evaluable["txgnn_disease_idx"] = txgnn_graph_present_evaluable["txgnn_disease_idx"].astype(int)

# Construct row-level positive edge table first.
# This preserves every original MiRAGE row before TxGNN-edge deduplication.
txgnn_external_test_positive_edges_raw = pd.DataFrame({
    "x_type": "drug",
    "x_id": txgnn_graph_present_evaluable["primekg_drug_node_id"].apply(normalize_primekg_node_id),
    "x_idx": txgnn_graph_present_evaluable["txgnn_drug_idx"].astype(int),
    "x_name": txgnn_graph_present_evaluable["DrugName"].fillna(
        txgnn_graph_present_evaluable["primekg_drug_node_name"]
    ),

    "relation": DEFAULT_EXTERNAL_RELATION,

    "y_type": "disease",
    "y_id": txgnn_graph_present_evaluable["primekg_disease_node_id"].apply(normalize_primekg_node_id),
    "y_idx": txgnn_graph_present_evaluable["txgnn_disease_idx"].astype(int),
    "y_name": txgnn_graph_present_evaluable["DiseaseName"].fillna(
        txgnn_graph_present_evaluable["primekg_disease_node_name"]
    ),

    "label": 1,
    "source": "MiRAGE",

    # Keep original identifiers for traceability
    "original_DrugID": txgnn_graph_present_evaluable["DrugID"],
    "original_DiseaseID": txgnn_graph_present_evaluable["DiseaseID"],
    "original_DrugName": txgnn_graph_present_evaluable["DrugName"],
    "original_DiseaseName": txgnn_graph_present_evaluable["DiseaseName"],
})

txgnn_external_test_positive_edges_raw = txgnn_external_test_positive_edges_raw[
    [
        "x_type",
        "x_id",
        "x_idx",
        "x_name",
        "relation",
        "y_type",
        "y_id",
        "y_idx",
        "y_name",
        "label",
        "source",
        "original_DrugID",
        "original_DiseaseID",
        "original_DrugName",
        "original_DiseaseName",
    ]
]

# ============================================================
# Inspect duplicate TxGNN edges
# ============================================================

edge_key_cols = ["x_type", "x_idx", "relation", "y_type", "y_idx"]

dup_mask = txgnn_external_test_positive_edges_raw.duplicated(
    subset=edge_key_cols,
    keep=False,
)

n_dup_rows = int(dup_mask.sum())
n_dup_extra = int(txgnn_external_test_positive_edges_raw.duplicated(
    subset=edge_key_cols,
    keep="first",
).sum())

print("TxGNN external positive edge table summary before dedup:")
print(f"Graph-present unseen rows: {len(mirage_graph_present_unseen):,}")
print(f"TxGNN evaluable raw rows: {len(txgnn_external_test_positive_edges_raw):,}")
print(f"Rows involved in duplicated TxGNN edges: {n_dup_rows:,}")
print(f"Duplicate extra rows to remove: {n_dup_extra:,}")

if n_dup_rows > 0:
    print("\nExample duplicated TxGNN edges:")
    duplicated_examples = (
        txgnn_external_test_positive_edges_raw.loc[dup_mask]
        .sort_values(edge_key_cols)
        .head(30)
    )
    display(duplicated_examples)

    print("\nDuplicate groups summary example:")
    dup_group_summary = (
        txgnn_external_test_positive_edges_raw.loc[dup_mask]
        .groupby(edge_key_cols)
        .agg(
            n_rows=("source", "size"),
            x_id=("x_id", "first"),
            y_id=("y_id", "first"),
            x_name=("x_name", "first"),
            y_name=("y_name", "first"),
            original_DiseaseIDs=("original_DiseaseID", lambda s: "|".join(sorted(set(map(str, s))))),
            original_DiseaseNames=("original_DiseaseName", lambda s: "|".join(sorted(set(map(str, s))))),
        )
        .reset_index()
        .sort_values("n_rows", ascending=False)
        .head(20)
    )
    display(dup_group_summary)

# ============================================================
# Deduplicate for GNN positive edge evaluation
# ============================================================
# For GNN edge scoring, duplicated (drug local idx, disease local idx, relation)
# edges are identical test edges. Keep one representative row.
txgnn_external_test_positive_edges = (
    txgnn_external_test_positive_edges_raw
    .drop_duplicates(subset=edge_key_cols, keep="first")
    .copy()
    .reset_index(drop=True)
)

# Also create an audit table containing all raw rows that collapsed to the same TxGNN edge.
txgnn_external_test_duplicate_edges = (
    txgnn_external_test_positive_edges_raw.loc[dup_mask]
    .copy()
    .reset_index(drop=True)
)

# ============================================================
# Final summary and sanity checks
# ============================================================

print("\nTxGNN external positive edge table summary after dedup:")
print(f"Raw positive rows: {len(txgnn_external_test_positive_edges_raw):,}")
print(f"Deduplicated positive edges: {len(txgnn_external_test_positive_edges):,}")
print(f"Removed duplicate rows: {len(txgnn_external_test_positive_edges_raw) - len(txgnn_external_test_positive_edges):,}")

print("\nRelation distribution:")
print(txgnn_external_test_positive_edges["relation"].value_counts())

print("\nUnique counts after dedup:")
print("Unique drugs:", txgnn_external_test_positive_edges["x_id"].nunique())
print("Unique diseases:", txgnn_external_test_positive_edges["y_id"].nunique())
print(
    "Unique drug-disease edges:",
    txgnn_external_test_positive_edges[["x_id", "y_id"]].drop_duplicates().shape[0],
)

print("\nDeduplicated edge table head:")
display(txgnn_external_test_positive_edges.head())

# Sanity checks
assert (txgnn_external_test_positive_edges["x_type"] == "drug").all()
assert (txgnn_external_test_positive_edges["y_type"] == "disease").all()
assert (txgnn_external_test_positive_edges["relation"] == DEFAULT_EXTERNAL_RELATION).all()
assert (txgnn_external_test_positive_edges["label"] == 1).all()
assert txgnn_external_test_positive_edges["x_idx"].notna().all()
assert txgnn_external_test_positive_edges["y_idx"].notna().all()

dup_edges_after = txgnn_external_test_positive_edges.duplicated(
    subset=edge_key_cols
).sum()
print(f"\nDuplicated TxGNN positive edges after dedup: {dup_edges_after:,}")
assert dup_edges_after == 0

print("\nSanity checks passed.")

# ============================================================
# 14. Save final MiRAGE external split outputs
# ============================================================

mirage_graph_present_to_save = (
    mirage_graph_present_unseen_with_txgnn_idx.copy()
    if "mirage_graph_present_unseen_with_txgnn_idx" in globals()
    else mirage_graph_present_unseen.copy()
)
mirage_graph_unpresent_to_save = mirage_graph_absent_unseen.copy()

removed_legacy_output_paths = remove_legacy_mirage_output_files()
summary = build_mirage_external_split_summary(mirage_graph_present_to_save)

mirage_graph_present_to_save.to_csv(FINAL_GRAPH_PRESENT_PATH, index=False)
mirage_graph_unpresent_to_save.to_csv(FINAL_GRAPH_UNPRESENT_PATH, index=False)

with open(FINAL_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

if removed_legacy_output_paths:
    print(f"\nRemoved legacy output files: {len(removed_legacy_output_paths):,}")
else:
    print("\nRemoved legacy output files: 0")

print("\nSaved final MiRAGE external split outputs:")
print(f"- {FINAL_SUMMARY_PATH} (keys={len(summary)})")
print(f"- {FINAL_GRAPH_PRESENT_PATH} shape={mirage_graph_present_to_save.shape}")
print(f"- {FINAL_GRAPH_UNPRESENT_PATH} shape={mirage_graph_unpresent_to_save.shape}")


TxGNN external positive edge table summary before dedup:
Graph-present unseen rows: 19,790
TxGNN evaluable raw rows: 19,790
Rows involved in duplicated TxGNN edges: 74
Duplicate extra rows to remove: 37

Example duplicated TxGNN edges:


,x_type,x_id,x_idx,x_name,relation,y_type,y_id,y_idx,y_name,label,source,original_DrugID,original_DiseaseID,original_DrugName,original_DiseaseName
8213,drug,DB00006,5,Bivalirudin,indication,disease,5053,12683,Ischemia,1,MiRAGE,DB00006,MESH:D007511,Bivalirudin,Ischemia
18813,drug,DB00006,5,Bivalirudin,indication,disease,5053,12683,Acute Coronary Syndrome,1,MiRAGE,DB00006,MESH:D054058,Bivalirudin,Acute Coronary Syndrome
3694,drug,DB00264,252,Metoprolol,indication,disease,5010,12650,Coronary Artery Disease,1,MiRAGE,DB00264,MESH:D003324,Metoprolol,Coronary Artery Disease
17170,drug,DB00264,252,Metoprolol,indication,disease,5010,12650,Myocardial Ischemia,1,MiRAGE,DB00264,MESH:D017202,Metoprolol,Myocardial Ischemia
3697,drug,DB00335,322,Atenolol,indication,disease,5010,12650,Coronary Artery Disease,1,MiRAGE,DB00335,MESH:D003324,Atenolol,Coronary Artery Disease
17165,drug,DB00335,322,Atenolol,indication,disease,5010,12650,Myocardial Ischemia,1,MiRAGE,DB00335,MESH:D017202,Atenolol,Myocardial Ischemia
3684,drug,DB00343,330,Diltiazem,indication,disease,5010,12650,Coronary Artery Disease,1,MiRAGE,DB00343,MESH:D003324,Diltiazem,Coronary Artery Disease
17089,drug,DB00343,330,Diltiazem,indication,disease,5010,12650,Myocardial Ischemia,1,MiRAGE,DB00343,MESH:D017202,Diltiazem,Myocardial Ischemia
3710,drug,DB00353,340,Methylergometrine,indication,disease,5010,12650,Coronary Artery Disease,1,MiRAGE,DB00353,MESH:D003324,Methylergometrine,Coronary Artery Disease
17122,drug,DB00353,340,Methylergometrine,indication,disease,5010,12650,Myocardial Ischemia,1,MiRAGE,DB00353,MESH:D017202,Methylergometrine,Myocardial Ischemia



Duplicate groups summary example:


,x_type,x_idx,relation,y_type,y_idx,n_rows,x_id,y_id,x_name,y_name,original_DiseaseIDs,original_DiseaseNames
0,drug,5,indication,disease,12683,2,DB00006,5053,Bivalirudin,Ischemia,MESH:D007511|MESH:D054058,Acute Coronary Syndrome|Ischemia
19,drug,710,indication,disease,12650,2,DB00727,5010,Nitroglycerin,Coronary Artery Disease,MESH:D003324|MESH:D017202,Coronary Artery Disease|Myocardial Ischemia
21,drug,740,indication,disease,12650,2,DB00758,5010,Clopidogrel,Coronary Artery Disease,MESH:D003324|MESH:D017202,Coronary Artery Disease|Myocardial Ischemia
22,drug,822,indication,disease,12650,2,DB00841,5010,Dobutamine,Coronary Artery Disease,MESH:D003324|MESH:D017202,Coronary Artery Disease|Myocardial Ischemia
23,drug,888,indication,disease,12502,2,DB00907,4849,Cocaine,Emphysema,MESH:D004646|MESH:D011656,Emphysema|Pulmonary Emphysema
24,drug,888,indication,disease,12650,2,DB00907,5010,Cocaine,Coronary Artery Disease,MESH:D003324|MESH:D017202,Coronary Artery Disease|Myocardial Ischemia
25,drug,888,indication,disease,12683,2,DB00907,5053,Cocaine,Ischemia,MESH:D007511|MESH:D054058,Acute Coronary Syndrome|Ischemia
26,drug,888,indication,disease,12888,2,DB00907,5294,Cocaine,Helicobacter Infections,MESH:D016481|MESH:D016491,Helicobacter Infections|Peripheral Vascular Di...
27,drug,896,indication,disease,13373,2,DB00915,5812,Amantadine,"Influenza, Human",MESH:D007251|MESH:D009976,"Influenza, Human|Orthomyxoviridae Infections"
28,drug,926,indication,disease,12650,2,DB00945,5010,Aspirin,Coronary Artery Disease,MESH:D003324|MESH:D017202,Coronary Artery Disease|Myocardial Ischemia



TxGNN external positive edge table summary after dedup:
Raw positive rows: 19,790
Deduplicated positive edges: 19,753
Removed duplicate rows: 37

Relation distribution:
relation
indication    19753
Name: count, dtype: int64

Unique counts after dedup:
Unique drugs: 1303
Unique diseases: 844
Unique drug-disease edges: 19753

Deduplicated edge table head:


,x_type,x_id,x_idx,x_name,relation,y_type,y_id,y_idx,y_name,label,source,original_DrugID,original_DiseaseID,original_DrugName,original_DiseaseName
0,drug,DB09140,5819,Oxygen,indication,disease,839,15671,Congenital Abnormalities,1,MiRAGE,DB09140,MESH:D000013,Oxygen,Congenital Abnormalities
1,drug,DB00730,712,Thiabendazole,indication,disease,839,15671,Congenital Abnormalities,1,MiRAGE,DB00730,MESH:D000013,Thiabendazole,Congenital Abnormalities
2,drug,DB00898,879,Ethanol,indication,disease,839,15671,Congenital Abnormalities,1,MiRAGE,DB00898,MESH:D000013,Ethanol,Congenital Abnormalities
3,drug,DB01168,1145,Procarbazine,indication,disease,839,15671,Congenital Abnormalities,1,MiRAGE,DB01168,MESH:D000013,Procarbazine,Congenital Abnormalities
4,drug,DB00550,534,Propylthiouracil,indication,disease,839,15671,Congenital Abnormalities,1,MiRAGE,DB00550,MESH:D000013,Propylthiouracil,Congenital Abnormalities



Duplicated TxGNN positive edges after dedup: 0

Sanity checks passed.
